# Download JSON Files from URLs

This notebook downloads JSON files from provided URLs and saves them to a specified output directory.

**Use Cases:**
- Download hotel data from supplier APIs
- Batch download JSON files from multiple sources
- Save API responses to local storage for processing

## 1. Import Required Libraries

In [1]:
import requests
import json
import os
from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime
import time

## 2. Configuration

Set your URLs and output directory here:

In [2]:
# List of URLs to download JSON files from
urls = [
    "https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/AE/1","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/1","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/2","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/3","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/4","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/5","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/6","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/7","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/1","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/2","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/3","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/4","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/5","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/6","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/7","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/8","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/9","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/10","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/11","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/12","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/13","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/14","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/15","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/16","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/17","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/18","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/19","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/20","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/21","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/22","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/23","https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/US/24"
]

# Output directory where JSON files will be saved
output_directory = "/Users/nakul.patil/Documents/hotel-match/expedia_input_data"

# Optional: Custom filenames (if empty, will use URL-based names)
custom_filenames = []

# Request timeout in seconds
timeout = 30

# Delay between requests (seconds) to be nice to servers
delay_between_requests = 1

## 3. Download Functions

In [3]:
def create_output_directory(output_dir):
    """Create output directory if it doesn't exist"""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    print(f"✓ Output directory ready: {output_dir}")


def get_filename_from_url(url, index):
    """Generate a filename from URL or use index"""
    parsed = urlparse(url)
    # Try to get filename from URL path
    path_parts = parsed.path.strip('/').split('/')
    if path_parts and path_parts[-1].endswith('.json'):
        return path_parts[-1]
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"download_{index}_{timestamp}.json"


def download_json_file(url, output_path, timeout=30):
    """
    Download a JSON file from URL and save to output_path
    
    Returns:
        tuple: (success: bool, message: str, data: dict or None)
    """
    try:
        print(f"Downloading: {url}")
        
        # Make HTTP request
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Raise exception for bad status codes
        
        # Try to parse as JSON
        try:
            json_data = response.json()
        except json.JSONDecodeError:
            return False, "Response is not valid JSON", None
        
        # Save to file
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
        
        file_size = os.path.getsize(output_path)
        return True, f"Successfully downloaded ({file_size:,} bytes)", json_data
        
    except requests.exceptions.Timeout:
        return False, f"Request timed out after {timeout} seconds", None
    except requests.exceptions.RequestException as e:
        return False, f"Request failed: {str(e)}", None
    except Exception as e:
        return False, f"Error: {str(e)}", None


def download_all_files(urls, output_dir, custom_names=None, timeout=30, delay=1):
    """
    Download multiple JSON files from URLs
    
    Args:
        urls: List of URLs to download
        output_dir: Directory to save files
        custom_names: Optional list of custom filenames
        timeout: Request timeout in seconds
        delay: Delay between requests in seconds
    
    Returns:
        dict: Summary of downloads
    """
    create_output_directory(output_dir)
    
    results = {
        'total': len(urls),
        'successful': 0,
        'failed': 0,
        'details': []
    }
    
    for i, url in enumerate(urls, 1):
        # Determine filename
        if custom_names and i-1 < len(custom_names) and custom_names[i-1]:
            filename = custom_names[i-1]
            if not filename.endswith('.json'):
                filename += '.json'
        else:
            filename = get_filename_from_url(url, i)
        
        output_path = os.path.join(output_dir, filename)
        
        # Download file
        print(f"\n[{i}/{len(urls)}] Processing: {filename}")
        success, message, data = download_json_file(url, output_path, timeout)
        
        # Record result
        result = {
            'url': url,
            'filename': filename,
            'success': success,
            'message': message,
            'path': output_path if success else None
        }
        results['details'].append(result)
        
        if success:
            results['successful'] += 1
            print(f"  ✓ {message}")
            print(f"  → Saved to: {output_path}")
        else:
            results['failed'] += 1
            print(f"  ✗ {message}")
        
        # Delay between requests (except after last one)
        if i < len(urls) and delay > 0:
            time.sleep(delay)
    
    return results

## 4. Execute Download

Run this cell to download all JSON files:

In [4]:
print("=" * 70)
print("JSON File Downloader")
print("=" * 70)
print(f"\nTotal URLs to process: {len(urls)}")
print(f"Output directory: {output_directory}\n")

# Execute downloads
results = download_all_files(
    urls=urls,
    output_dir=output_directory,
    custom_names=custom_filenames,
    timeout=timeout,
    delay=delay_between_requests
)

# Print summary
print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)
print(f"Total: {results['total']}")
print(f"✓ Successful: {results['successful']}")
print(f"✗ Failed: {results['failed']}")
print("=" * 70)

JSON File Downloader

Total URLs to process: 32
Output directory: /Users/nakul.patil/Documents/hotel-match/expedia_input_data

✓ Output directory ready: /Users/nakul.patil/Documents/hotel-match/expedia_input_data

[1/32] Processing: download_1_20260209_162239.json
Downloading: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/AE/1
  ✓ Successfully downloaded (13,343,558 bytes)
  → Saved to: /Users/nakul.patil/Documents/hotel-match/expedia_input_data/download_1_20260209_162239.json

[2/32] Processing: download_2_20260209_162248.json
Downloading: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/1
  ✓ Successfully downloaded (12,746,002 bytes)
  → Saved to: /Users/nakul.patil/Documents/hotel-match/expedia_input_data/download_2_20260209_162248.json

[3/32] Processing: download_3_20260209_162303.json
Downloading: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/2
  ✓ Successfully downloaded (12,699,076 bytes)
  → Saved to: 

## 5. View Download Details

Inspect the results to see which files were downloaded successfully:

In [5]:
import pandas as pd

# Create DataFrame from results
df_results = pd.DataFrame(results['details'])

# Display results
print("\nDetailed Results:")
print("-" * 100)
for idx, row in df_results.iterrows():
    status = "✓" if row['success'] else "✗"
    print(f"{status} {row['filename']}")
    print(f"  URL: {row['url']}")
    print(f"  Status: {row['message']}")
    if row['path']:
        print(f"  Path: {row['path']}")
    print()

# Show summary table
print("\nSummary Table:")
df_results[['filename', 'success', 'message']].style.set_properties(**{'text-align': 'left'})


Detailed Results:
----------------------------------------------------------------------------------------------------
✓ download_1_20260209_162239.json
  URL: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/AE/1
  Status: Successfully downloaded (13,343,558 bytes)
  Path: /Users/nakul.patil/Documents/hotel-match/expedia_input_data/download_1_20260209_162239.json

✓ download_2_20260209_162248.json
  URL: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/1
  Status: Successfully downloaded (12,746,002 bytes)
  Path: /Users/nakul.patil/Documents/hotel-match/expedia_input_data/download_2_20260209_162248.json

✓ download_3_20260209_162303.json
  URL: https://zh-content-data.s3.us-east-1.amazonaws.com/EAN/20251010/en-US/IN/2
  Status: Successfully downloaded (12,699,076 bytes)
  Path: /Users/nakul.patil/Documents/hotel-match/expedia_input_data/download_3_20260209_162303.json

✓ download_4_20260209_162321.json
  URL: https://zh-content-data.s3.us

,filename,success,message
0,download_1_20260209_162239.json,True,"Successfully downloaded (13,343,558 bytes)"
1,download_2_20260209_162248.json,True,"Successfully downloaded (12,746,002 bytes)"
2,download_3_20260209_162303.json,True,"Successfully downloaded (12,699,076 bytes)"
3,download_4_20260209_162321.json,True,"Successfully downloaded (12,722,216 bytes)"
4,download_5_20260209_162327.json,True,"Successfully downloaded (12,737,454 bytes)"
5,download_6_20260209_162333.json,True,"Successfully downloaded (12,706,746 bytes)"
6,download_7_20260209_162358.json,True,"Successfully downloaded (12,725,410 bytes)"
7,download_8_20260209_162418.json,True,"Successfully downloaded (6,408,998 bytes)"
8,download_9_20260209_162424.json,True,"Successfully downloaded (15,480,181 bytes)"
9,download_10_20260209_162447.json,True,"Successfully downloaded (15,510,917 bytes)"


## 6. Verify Downloaded Files

List all JSON files in the output directory:

In [6]:
# List downloaded files
if os.path.exists(output_directory):
    json_files = sorted([f for f in os.listdir(output_directory) if f.endswith('.json')])
    
    print(f"Found {len(json_files)} JSON file(s) in {output_directory}:\n")
    
    for filename in json_files:
        filepath = os.path.join(output_directory, filename)
        file_size = os.path.getsize(filepath)
        modified_time = datetime.fromtimestamp(os.path.getmtime(filepath))
        
        print(f"📄 {filename}")
        print(f"   Size: {file_size:,} bytes")
        print(f"   Modified: {modified_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Preview first few lines
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                if isinstance(data, list):
                    print(f"   Type: List with {len(data)} items")
                elif isinstance(data, dict):
                    print(f"   Type: Dictionary with {len(data)} keys")
                    print(f"   Keys: {list(data.keys())[:5]}")
        except Exception as e:
            print(f"   Error reading file: {e}")
        print()
else:
    print(f"Output directory does not exist: {output_directory}")

Found 32 JSON file(s) in /Users/nakul.patil/Documents/hotel-match/expedia_input_data:

📄 download_10_20260209_162447.json
   Size: 15,510,917 bytes
   Modified: 2026-02-09 16:24:56
   Type: List with 5000 items

📄 download_11_20260209_162457.json
   Size: 15,437,911 bytes
   Modified: 2026-02-09 16:25:08
   Type: List with 5000 items

📄 download_12_20260209_162509.json
   Size: 15,412,179 bytes
   Modified: 2026-02-09 16:25:19
   Type: List with 5000 items

📄 download_13_20260209_162520.json
   Size: 15,494,192 bytes
   Modified: 2026-02-09 16:25:59
   Type: List with 5000 items

📄 download_14_20260209_162600.json
   Size: 15,467,565 bytes
   Modified: 2026-02-09 16:26:05
   Type: List with 5000 items

📄 download_15_20260209_162606.json
   Size: 15,512,580 bytes
   Modified: 2026-02-09 16:26:11
   Type: List with 5000 items

📄 download_16_20260209_162612.json
   Size: 15,409,148 bytes
   Modified: 2026-02-09 16:26:33
   Type: List with 5000 items

📄 download_17_20260209_162634.json
   

## 7. Example: Preview JSON Content

Load and display content from a specific file:

In [7]:
# Select a file to preview (change index as needed)
if os.path.exists(output_directory):
    json_files = sorted([f for f in os.listdir(output_directory) if f.endswith('.json')])
    
    if json_files:
        # Preview first file
        preview_file = json_files[0]
        preview_path = os.path.join(output_directory, preview_file)
        
        print(f"Previewing: {preview_file}\n")
        print("-" * 70)
        
        with open(preview_path, 'r') as f:
            data = json.load(f)
        
        # Pretty print (limit to first few items if list)
        if isinstance(data, list):
            print(f"List with {len(data)} items\n")
            print("First 2 items:")
            print(json.dumps(data[:2], indent=2))
        else:
            print(json.dumps(data, indent=2)[:1000] + "\n...")
    else:
        print("No JSON files found in output directory")
else:
    print(f"Output directory does not exist: {output_directory}")

Previewing: download_10_20260209_162447.json

----------------------------------------------------------------------
List with 5000 items

First 2 items:
[
  {
    "id": "71575314",
    "name": "Zenfulcove",
    "relevanceScore": 0,
    "providerId": "EAN",
    "providerHotelId": "109207676",
    "languageCode": "en-US",
    "providerName": "EAN",
    "geoCode": {
      "lat": 30.308914,
      "long": -97.2819
    },
    "neighbourhoods": null,
    "contact": {
      "address": {
        "line1": "103 Potato Smith Rd",
        "line2": "Unit C",
        "destinationCode": null,
        "city": {
          "code": null,
          "name": "Elgin"
        },
        "state": {
          "code": "TX",
          "name": "Texas"
        },
        "country": {
          "code": "US",
          "name": "United States Of America"
        },
        "postalCode": "78621"
      },
      "phones": [
        "1-8329701236"
      ],
      "fax": [
        null
      ],
      "emails": null,
      "